# **Machine Learning**

### **Import**

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from pymongo import MongoClient
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

load_dotenv()
db = MongoClient(os.getenv("MONGO_URI"))["cinema"]

In [2]:
films = pd.DataFrame(db["films"].find(
    {"budget": {"$gte": 10_000}, "recettes": {"$gte": 10_000}},
    {"titre": 1, "budget": 1, "recettes": 1, "duree_min": 1, "genres": 1,
     "date_sortie": 1, "annee": 1, "langue_originale": 1, "saga": 1}
))
films = films.dropna(subset=["duree_min", "date_sortie"])
print(len(films), "films")
print("Colonnes :", list(films.columns))

4185 films
Colonnes : ['_id', 'titre', 'langue_originale', 'date_sortie', 'duree_min', 'budget', 'recettes', 'genres', 'annee', 'saga']


In [3]:
films["log_recettes"] = np.log10(films["recettes"])
films["log_budget"] = np.log10(films["budget"])
films["mois"] = pd.to_datetime(films["date_sortie"]).dt.month
films["saga"] = films["saga"].notna().astype(int)
films["anglais"] = (films["langue_originale"] == "en").astype(int)

genres = films["genres"].str.join("|").str.get_dummies()
genres.columns = ["genre_" + g for g in genres.columns]
films = pd.concat([films, genres], axis=1)

variables = ["log_budget", "duree_min", "mois", "annee", "saga", "anglais"] + list(genres.columns)
print(len(variables), "variables explicatives")

25 variables explicatives


In [4]:
X = films[variables]
y = films["log_recettes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Entraînement :", len(X_train), "films | Test :", len(X_test), "films")

Entraînement : 3348 films | Test : 837 films


In [5]:
modeles = {
    "Régression linéaire": LinearRegression(),
    "Forêt aléatoire": RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                             random_state=42, n_jobs=-1),
}
scores = []
for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    pred = modele.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    scores.append({"modèle": nom, "R²": round(r2_score(y_test, pred), 3),
                   "erreur typique (×)": round(10 ** mae, 2)})
pd.DataFrame(scores)

,modèle,R²,erreur typique (×)
0,Régression linéaire,0.497,2.48
1,Forêt aléatoire,0.494,2.48


In [6]:
print("Variables suspectes :", [v for v in variables if "recette" in v or "vote" in v or "note" in v])
print(X.corrwith(y).abs().sort_values(ascending=False).head(5))

Variables suspectes : []
log_budget        0.663539
saga              0.347882
genre_Aventure    0.308252
genre_Drame       0.234937
duree_min         0.229411
dtype: float64


### **Matrice de corrélations**

In [9]:
# On garde les variables principales et quelques genres (les 19 genres rendraient la matrice illisible)
colonnes = ["log_recettes", "log_budget", "duree_min", "mois", "annee", "saga", "anglais",
            "genre_Aventure", "genre_Animation", "genre_Action", "genre_Drame", "genre_Horreur"]

correlation = films[colonnes].corr()

fig = px.imshow(
    correlation,
    text_auto=".2f",                     # affiche la valeur dans chaque case
    color_continuous_scale="RdBu_r",     # rouge = lien positif, bleu = lien négatif
    zmin=-1, zmax=1,                     # échelle fixe de -1 à 1
    title="Matrice de corrélation des variables",
)
fig.update_layout(height=700)
fig.show()

- la première ligne (log_recettes) est la plus importante : elle montre le lien de chaque variable avec ce qu'on veut prédire ;
- les cases rouge foncé indiquent un lien positif fort, les cases bleu foncé un lien négatif fort ;
- la diagonale vaut toujours 1 : une variable est parfaitement liée à elle-même.

🖇️​Lien budget / recettes (fort et positif), et Drame et l'Horreur sont en bleu, c'est-à-dire associés à des recettes plus faibles.

In [11]:
importance = pd.Series(modeles["Forêt aléatoire"].feature_importances_, index=variables)
importance = importance.sort_values().tail(10)

fig = px.bar(importance, orientation="h",
             title="Les 10 variables les plus importantes pour prédire les recettes",
             labels={"value": "Importance", "index": ""})
fig.update_layout(showlegend=False)
fig.show()

In [12]:
modele = modeles["Régression linéaire"]
exemples = pd.DataFrame({
    "titre": films.loc[X_test.index, "titre"],
    "budget (M$)": (films.loc[X_test.index, "budget"] / 1e6).round(0),
    "prédit (M$)": (10 ** modele.predict(X_test) / 1e6).round(0),   # 10 ** … annule le log
    "réel (M$)": (10 ** y_test / 1e6).round(0),
})

# Les 10 films les plus connus du jeu de test (plus gros budgets)
exemples.sort_values("budget (M$)", ascending=False).head(10)

,titre,budget (M$),prédit (M$),réel (M$)
862,Deadpool & Wolverine,557.0,862.0,1338.0
240,Star Wars : L'Ascension de Skywalker,490.0,1041.0,1074.0
43,Avengers : Endgame,356.0,1467.0,2943.0
174,Star Wars : Les Derniers Jedi,300.0,850.0,1334.0
17,Avengers : Infinity War,300.0,879.0,2052.0
1024,F1® Le Film,300.0,686.0,634.0
1117,The Marvels,275.0,385.0,206.0
864,Solo: A Star Wars Story,250.0,246.0,393.0
665,John Carter,250.0,310.0,284.0
449,Thor : Love and Thunder,250.0,400.0,761.0


- Bien prédits : F1 (686 prédits, 634 réels), John Carter (310 contre 284), Star Wars : L'Ascension de Skywalker (1 041 contre 1 074). Le modèle tombe assez juste.
- Phénomènes sous-estimés : Avengers : Endgame (1 467 prédits, mais 2 943 réels) et Infinity War (879 contre 2 052). 
Le modèle ne peut pas savoir que ces films concluaient dix ans de saga Marvel, avec un engouement mondial exceptionnel.
- Flop surestimé : The Marvels (385 prédits, seulement 206 réels). Sur le papier, c'était un film Marvel au gros budget, mais le public n'a pas suivi.

C'est exactement ce que dit le R² de 0,5 : le modèle capte la logique générale (gros budget et saga donnent de grosses recettes), mais pas l'imprévisible (l'engouement, la lassitude du public, la qualité du film).